In [34]:
import pandas as pd
import altair as alt

In [24]:
file_path = '/Users/juanpazmino/Documents/Desarrollo/Personal_projects/data_visualization/Lab01/01mdi_homicidios_intencionales_pm_2014_2025.xlsx'
df = pd.read_excel(file_path, sheet_name='1', header=1)
df = df.drop(df.columns[0], axis=1)
df.head()

,tipo_muerte,zona,subzona,distrito,circuito,codigo_subcircuito,subcircuito,codigo_provincia,provincia,codigo_canton,...,medida_edad,sexo,genero,etnia,estado_civil,nacionalidad,discapacidad,profesion_registro_civil,instruccion,antecedentes
0,ASESINATO,ZONA 1,ESMERALDAS,ESMERALDAS,LAS PALMAS,08D01C02S01,LAS PALMAS 1,8,ESMERALDAS,801,...,A,MUJER,FEMENINO,AFRO,SOLTERO,ECUADOR,NINGUNA,ESTADO PERSONAL,SIN_DATO,SIN_DATO
1,ASESINATO,ZONA 4,MANABÍ,PORTOVIEJO,SAN PABLO,13D01C05S02,SAN PABLO 2,13,MANABÍ,1301,...,A,HOMBRE,MASCULINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,TRABAJADOR GENERAL,BASICA,SIN_DATO
2,ASESINATO,ZONA 4,MANABÍ,PORTOVIEJO,SAN PABLO,13D01C05S02,SAN PABLO 2,13,MANABÍ,1301,...,A,HOMBRE,MASCULINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,MAESTRO DE OBRA/CONSTRUCCIÓN,SIN_DATO,SIN_DATO
3,ASESINATO,ZONA 4,MANABÍ,MANTA,LA PILA,13D02C14S01,LA PILA 1,13,MANABÍ,1309,...,A,MUJER,FEMENINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,SIN_DATO,SIN_DATO,SIN_DATO
4,ASESINATO,ZONA 4,MANABÍ,MANTA,LA PILA,13D02C14S01,LA PILA 1,13,MANABÍ,1309,...,A,MUJER,FEMENINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,ESTADO PERSONAL,SECUNDARIA,SIN_DATO


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39773 entries, 0 to 39772
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   tipo_muerte               39773 non-null  str           
 1   zona                      39773 non-null  str           
 2   subzona                   39773 non-null  str           
 3   distrito                  39773 non-null  str           
 4   circuito                  39773 non-null  str           
 5   codigo_subcircuito        39773 non-null  str           
 6   subcircuito               39773 non-null  str           
 7   codigo_provincia          39773 non-null  int64         
 8   provincia                 39773 non-null  str           
 9   codigo_canton             39773 non-null  int64         
 10  canton                    39773 non-null  str           
 11  coordenada_y              39773 non-null  str           
 12  coordenada_x              397

In [30]:
date_str = pd.to_datetime(df['fecha_infraccion'], errors='coerce').dt.strftime('%Y-%m-%d')
time_str = df['hora_infraccion'].astype(str)

df['fecha_hora_infraccion'] = pd.to_datetime(
    date_str + ' ' + time_str,
    errors='coerce' 
)

df = df.drop(columns=['fecha_infraccion', 'hora_infraccion'])

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39773 entries, 0 to 39772
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   tipo_muerte               39773 non-null  str           
 1   zona                      39773 non-null  str           
 2   subzona                   39773 non-null  str           
 3   distrito                  39773 non-null  str           
 4   circuito                  39773 non-null  str           
 5   codigo_subcircuito        39773 non-null  str           
 6   subcircuito               39773 non-null  str           
 7   codigo_provincia          39773 non-null  int64         
 8   provincia                 39773 non-null  str           
 9   codigo_canton             39773 non-null  int64         
 10  canton                    39773 non-null  str           
 11  coordenada_y              39773 non-null  str           
 12  coordenada_x              397

# **Pregunta 1: Patrón temporal semanal**

¿Existen días de la semana donde ocurren más homicidios?
¿Se observa concentración en fines de semana o días laborales?

**Enfoque analítico esperado:**
Identificar periodicidad semanal y comparar intensidad por día.

In [ ]:
# 1. Extraer el día de la semana numérico (0 = Lunes, 6 = Domingo)
df['dia_semana_num'] = df['fecha_hora_infraccion'].dt.dayofweek

# 2. Mapear a nombres de días para facilitar la lectura
dias_map = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles',
    3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}
df['dia_semana'] = df['dia_semana_num'].map(dias_map)

# 3. Conteo de homicidios por día de la semana (garantizando el orden correcto de los días)
contagem_por_dia = df['dia_semana'].value_counts().reindex(list(dias_map.values()))
print("--- Conteo de Homicidios por Día de la Semana ---")
print(contagem_por_dia)

# 4. Agrupar los datos comparando Fin de Semana vs Día Laboral
df['tipo_dia'] = df['dia_semana_num'].apply(lambda x: 'Fin de Semana' if x >= 5 else 'Día Laboral')
contagem_tipo_dia = df['tipo_dia'].value_counts()
print("\n--- Conteo por Tipo de Día (Fin de Semana vs Día Laboral) ---")
print(contagem_tipo_dia)

# 5. Visualización gráfica de los resultados con Altair
chart_dias = alt.Chart(df_plot).mark_bar(cornerRadiusEnd=4).encode(
    x=alt.X('Día de la Semana:N', 
            sort=list(dias_map.values()), 
            title='', 
            axis=alt.Axis(labelAngle=-45, labelFontSize=12)),
    y=alt.Y('Número de Ocurrencias:Q', title='Número de Homicidios'),
    # Condición de color: Destacamos Sábado y Domingo en rojo suave, el resto en azul
    color=alt.condition(
        "datum['Día de la Semana'] == 'Sábado' || datum['Día de la Semana'] == 'Domingo'",
        alt.value('#e45756'),  # Color de Fin de Semana
        alt.value('#4c78a8')   # Color de Día Laboral
    ),
    tooltip=['Día de la Semana', 'Número de Ocurrencias']
).properties(
    title=alt.TitleParams('Distribución de Homicidios: Fines de semana vs Días Laborales', fontSize=16),
    width=600,
    height=400
)

# Añadir el número exacto sobre cada barra para lectura rápida
text_dias = chart_dias.mark_text(
    align='center',
    baseline='bottom',
    dy=-5,
    fontWeight='bold'
).encode(
    text='Número de Ocurrencias:Q'
)

(chart_dias + text_dias).show()

--- Conteo de Homicidios por Día de la Semana ---
dia_semana
Lunes        5313
Martes       4891
Miércoles    4830
Jueves       5023
Viernes      5577
Sábado       6599
Domingo      7515
Name: count, dtype: int64

--- Conteo por Tipo de Día (Fin de Semana vs Día Laboral) ---
tipo_dia
Día Laboral      25659
Fin de Semana    14114
Name: count, dtype: int64


alt.LayerChart(...)

# **Pregunta 2: Hora más crítica del día**

¿En qué horas del día se concentran más homicidios?
¿Existe un patrón nocturno, vespertino o matutino?

 
Enfoque analítico esperado:
Análisis de distribución temporal intra-diaria.

In [47]:
# 1. Extraer la hora del día (0-23)
df['hora_del_dia'] = df['fecha_hora_infraccion'].dt.hour

# 2. Conteo de homicidios por hora
conteo_por_hora = df['hora_del_dia'].value_counts().sort_index()
print("\n--- Conteo de Homicidios por Hora del Día ---")
print(conteo_por_hora)

# 3. Agrupar por franja horaria para ver el patrón (Madrugada, Mañana, Tarde, Noche)
def categorizar_franja_horaria(hora):
    if pd.isna(hora):
        return 'Desconocido'
    elif 0 <= hora < 6:
        return '1 - Madrugada (00:00 - 05:59)'
    elif 6 <= hora < 12:
        return '2 - Mañana (06:00 - 11:59)'
    elif 12 <= hora < 18:
        return '3 - Tarde (12:00 - 17:59)'
    else:
        return '4 - Noche (18:00 - 23:59)'

df['franja_horaria'] = df['hora_del_dia'].apply(categorizar_franja_horaria)
conteo_franja = df['franja_horaria'].value_counts().sort_index()
print("\n--- Conteo por Franja Horaria ---")
print(conteo_franja)

# 4. Visualización gráfica con Altair (Distribución por Hora)
area = alt.Chart(df_plot_hora).mark_area(
    opacity=0.6, 
    interpolate='monotone', 
    color='#F58518'
).encode(
    x=alt.X('Hora del Día:O', title='Hora del Día', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Número de Ocurrencias:Q', title='Número de Homicidios')
)

linea = alt.Chart(df_plot_hora).mark_line(
    interpolate='monotone', 
    color='#C95A00', 
    strokeWidth=3
).encode(
    x=alt.X('Hora del Día:O'),
    y=alt.Y('Número de Ocurrencias:Q'),
    tooltip=['Hora del Día', 'Número de Ocurrencias']
)

chart_hora = (area + linea).properties(
    title=alt.TitleParams('Evolución Intra-diaria de Homicidios', fontSize=16),
    width=650,
    height=350
)

chart_hora.show()


--- Conteo de Homicidios por Hora del Día ---
hora_del_dia
0.0     1638
1.0     1743
2.0     1508
3.0     1278
4.0      946
5.0      903
6.0      856
7.0     1007
8.0     1192
9.0     1182
10.0    1330
11.0    1395
12.0    1601
13.0    1580
14.0    1564
15.0    1707
16.0    1814
17.0    2022
18.0    2005
19.0    2279
20.0    2402
21.0    2606
22.0    2495
23.0    2695
Name: count, dtype: int64

--- Conteo por Franja Horaria ---
franja_horaria
1 - Madrugada (00:00 - 05:59)     8016
2 - Mañana (06:00 - 11:59)        6962
3 - Tarde (12:00 - 17:59)        10288
4 - Noche (18:00 - 23:59)        14482
Desconocido                         25
Name: count, dtype: int64


alt.LayerChart(...)

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39773 entries, 0 to 39772
Data columns (total 38 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   tipo_muerte               39773 non-null  str           
 1   zona                      39773 non-null  str           
 2   subzona                   39773 non-null  str           
 3   distrito                  39773 non-null  str           
 4   circuito                  39773 non-null  str           
 5   codigo_subcircuito        39773 non-null  str           
 6   subcircuito               39773 non-null  str           
 7   codigo_provincia          39773 non-null  int64         
 8   provincia                 39773 non-null  str           
 9   codigo_canton             39773 non-null  int64         
 10  canton                    39773 non-null  str           
 11  coordenada_y              39773 non-null  str           
 12  coordenada_x              397

***
# **Pregunta 3: Evolución mensual de homicidios**

¿Cómo varían los homicidios a lo largo de los meses del año 2025?
¿Se observa una tendencia creciente, decreciente o estacional?

Enfoque analítico esperado:
Detección de tendencias y cambios en el tiempo.

In [43]:
# 1. Filtrar los datos solo para el año 2025
df_2025 = df[df['fecha_hora_infraccion'].dt.year == 2025].copy()

# 2. Extraer el mes numérico
df_2025['mes_num'] = df_2025['fecha_hora_infraccion'].dt.month.astype(int)

# 3. Mapear a los nombres de los meses en español
meses_map = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
    7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
df_2025['mes'] = df_2025['mes_num'].map(meses_map)

# 4. Conteo de homicidios por mes (manteniendo el orden cronológico)
conteo_mensual = df_2025['mes'].value_counts().reindex(list(meses_map.values())).dropna()
print("\n--- Conteo de Homicidios por Mes en 2025 ---")
print(conteo_mensual)

# 5. Visualización gráfica con Altair (Gráfico de líneas para ver tendencias)
df_plot_mes = conteo_mensual.reset_index()
df_plot_mes.columns = ['Mes', 'Número de Ocurrencias']

# Añadir una columna numérica para que Altair sepa cómo ordenar cronológicamente
df_plot_mes['Mes_Num'] = df_plot_mes['Mes'].map({v: k for k, v in meses_map.items()})

chart_mes = alt.Chart(df_plot_mes).mark_line(point=True).encode(
    x=alt.X('Mes:N', 
            sort=alt.EncodingSortField(field='Mes_Num', order='ascending'), 
            title='Mes (2025)', 
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('Número de Ocurrencias:Q', 
            title='Número de Ocurrencias', 
            scale=alt.Scale(zero=False)), # `zero=False` permite ver mejor las variaciones pequeñas
    tooltip=['Mes', 'Número de Ocurrencias']
).properties(
    title='Evolución Mensual de Homicidios en el Año 2025',
    width=600,
    height=400
)

chart_mes.show()


--- Conteo de Homicidios por Mes en 2025 ---
mes
Enero         800
Febrero       750
Marzo         857
Abril         741
Mayo          935
Junio         574
Julio         637
Agosto        767
Septiembre    766
Octubre       768
Noviembre     848
Diciembre     792
Name: count, dtype: int64


alt.Chart(...)

***
# **Pregunta 4: Diferencias sociodemográficas**

¿Existen diferencias relevantes en los homicidios según sexo y edad?
¿Qué grupo poblacional presenta mayor concentración de casos?

Enfoque analítico esperado:
Comparación entre grupos poblacionales y distribución de frecuencias.

In [44]:
# 1. Limpieza de la columna 'edad'
# Convertimos a numérico porque en el df.info() aparece como 'object' (texto).
# errors='coerce' transformará cualquier valor no numérico en NaN de forma segura.
df['edad_num'] = pd.to_numeric(df['edad'], errors='coerce')

# 2. Análisis por Sexo
conteo_sexo = df['sexo'].value_counts()
print("\n--- Conteo de Homicidios por Sexo ---")
print(conteo_sexo)
print("\n--- Porcentaje por Sexo (%) ---")
print((conteo_sexo / conteo_sexo.sum() * 100).round(2))

# 3. Análisis por Rango de Edad
# Definimos los límites (bins) y las etiquetas para los grupos poblacionales
limites_edad = [0, 17, 29, 44, 59, 120]
etiquetas_edad = ['0-17 (Menores)', '18-29 (Jóvenes)', '30-44 (Adultos)', '45-59 (Maduros)', '60+ (3ra Edad)']

df['rango_edad'] = pd.cut(df['edad_num'], bins=limites_edad, labels=etiquetas_edad, right=True)

conteo_edad = df['rango_edad'].value_counts().sort_index()
print("\n--- Conteo de Homicidios por Rango de Edad ---")
print(conteo_edad)

# 4. Visualización Gráfica Combinada con Altair

# A. Gráfico de Dona para la proporción por Sexo
df_plot_sexo = conteo_sexo.reset_index()
df_plot_sexo.columns = ['Sexo', 'Número de Ocurrencias']

chart_sexo = alt.Chart(df_plot_sexo).mark_arc(innerRadius=50).encode(
    theta=alt.Theta(field='Número de Ocurrencias', type='quantitative'),
    color=alt.Color(field='Sexo', type='nominal', scale=alt.Scale(scheme='set1')),
    tooltip=['Sexo', 'Número de Ocurrencias']
).properties(
    title='Proporción por Sexo',
    width=250,
    height=300
)

# B. Gráfico de Barras Agrupadas para Rango de Edad segmentado por Sexo
# Agrupamos por ambas variables para graficarlas juntas
df_plot_demografia = df.dropna(subset=['rango_edad', 'sexo']).groupby(['rango_edad', 'sexo']).size().reset_index(name='Número de Ocurrencias')

chart_demografia = alt.Chart(df_plot_demografia).mark_bar().encode(
    x=alt.X('rango_edad:O', title='Rango de Edad', axis=alt.Axis(labelAngle=-25)),
    y=alt.Y('Número de Ocurrencias:Q', title='Número de Ocurrencias'),
    color=alt.Color('sexo:N', title='Sexo', scale=alt.Scale(scheme='set1')),
    xOffset='sexo:N', # Esto crea el efecto de barras agrupadas (lado a lado)
    tooltip=['rango_edad', 'sexo', 'Número de Ocurrencias']
).properties(
    title='Distribución por Rango de Edad y Sexo',
    width=500,
    height=300
)

# C. Mostrar ambos gráficos uno al lado del otro
chart_combinado = chart_sexo | chart_demografia
chart_combinado.show()


--- Conteo de Homicidios por Sexo ---
sexo
HOMBRE            35868
MUJER              3740
NO DETERMINADO      164
SIN_DATO              1
Name: count, dtype: int64

--- Porcentaje por Sexo (%) ---
sexo
HOMBRE            90.18
MUJER              9.40
NO DETERMINADO     0.41
SIN_DATO           0.00
Name: count, dtype: float64

--- Conteo de Homicidios por Rango de Edad ---
rango_edad
0-17 (Menores)      1995
18-29 (Jóvenes)    16001
30-44 (Adultos)    14600
45-59 (Maduros)     4659
60+ (3ra Edad)      1598
Name: count, dtype: int64


alt.HConcatChart(...)